In [1]:
import warnings
warnings.simplefilter(action='ignore')

import os
import requests
import joblib

import numpy, pandas
import pandas as pd

import matplotlib.pyplot as plt
import scipy.stats as stats

In [2]:
from catboost import CatBoostRegressor ,Pool
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

from sklearn.linear_model import ElasticNetCV
from sklearn.linear_model import LassoCV
from sklearn.linear_model import RidgeCV
from sklearn.linear_model import LassoLars
from sklearn.linear_model import Lasso
from sklearn.linear_model import BayesianRidge
from sklearn.linear_model import TweedieRegressor
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LogisticRegression

from sklearn.neural_network import MLPRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.cross_decomposition import PLSRegression
from sklearn.cluster import KMeans
from sklearn.neighbors import NearestCentroid

from sklearn.naive_bayes import GaussianNB
from sklearn.naive_bayes import MultinomialNB
from sklearn.naive_bayes import BernoulliNB
from sklearn.naive_bayes import CategoricalNB

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import ExtraTreesRegressor

from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import BaggingRegressor
from sklearn.ensemble import VotingRegressor
from sklearn.ensemble import StackingRegressor

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import PolynomialFeatures
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler

from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import accuracy_score

from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold, cross_val_score

from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_classif
from sklearn.feature_selection import SelectFromModel

from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.model_selection import cross_val_score

In [3]:
os.makedirs('data', exist_ok=True)
os.makedirs('model', exist_ok=True)

In [4]:
train = pandas.read_csv('/kaggle/input/zelestra-phase-2-data/Dataset 1.csv')

In [5]:
train_coor = pandas.read_csv('/kaggle/input/zelestra-phase-2-data/all_cor.csv').fillna(0)
index = [train_coor[train_coor['Unnamed: 0'] == i].index for i in ['ppc_consig_p', 'ppc_p_tot', 'ppc_eact_export'
                                                                   , 'ppc_eact_imp']]
for i in index:
    train_coor = train_coor.drop(i)
columns = numpy.array(train_coor[train_coor['ttr_potenciaproducible'] > 0.4]['Unnamed: 0'])
train = train[columns]

In [6]:
encoder = LabelEncoder()
normalize = SimpleImputer(strategy='mean')

for i in zip(train.columns, train.dtypes):
    if i[1]=='O':
        train[i[0]] = train[i[0]].fillna('Unknown')
        train[i[0]] = normalize.fit_transform(encoder.fit_transform(train[i[0]].to_numpy().reshape(-1,1)).reshape(-1,1))
    else:
        train[i[0]].fillna(train[i[0]].abs().mean(), inplace=True)
        train[i[0]] = normalize.fit_transform(numpy.array(train[i[0]].abs()).reshape(-1,1))

In [7]:
train_c_x, train_c_y = train.drop(columns=['ttr_potenciaproducible']), train['ttr_potenciaproducible']

In [8]:
train_c_x=train_c_x.reset_index().drop(columns=['index'])
train_c_y=train_c_y.reset_index().drop(columns=['index'])

In [9]:
train_c_x

,meteorolgicas_em_03_02_gii,meteorolgicas_em_08_01_gii,meteorolgicas_em_03_02_ghi,meteorolgicas_em_08_01_ghi,meteorolgicas_em_08_01_gii_rear,meteorolgicas_em_03_02_gii_rear,meteorolgicas_em_03_02_t_amb,meteorolgicas_em_08_01_t_amb,meteorolgicas_em_03_02_t_dlogger,meteorolgicas_em_08_01_t_dlogger,...,inversores_ctin08_strings_string12_pv_i2,inversores_ctin08_strings_string12_pv_i8,inversores_ctin08_strings_string12_pv_i9,inversores_ctin08_strings_string12_pv_i10,inversores_ctin08_inv_08_08_p,inversores_ctin03_inv_03_03_p,inversores_ctin03_inv_03_03_p_dc,inversores_ctin08_inv_08_08_p_dc,inversores_ctin03_inv_03_03_eact_tot,inversores_ctin08_inv_08_08_eact_tot
0,0.0,0.0,0.0,0.0,0.0,0.0,19.034977,18.220884,19.073761,18.246216,...,6.772808,7.475758,7.639425,7.313804,0.0,0.0,0.0,0.0,0.148267,0.146776
1,0.0,0.0,0.0,0.0,0.0,0.0,18.737464,18.397779,18.972198,18.192261,...,6.772808,7.475758,7.639425,7.313804,0.0,0.0,0.0,0.0,0.000000,0.000000
2,0.0,0.0,0.0,0.0,0.0,0.0,18.681954,18.268321,18.871460,18.188324,...,6.772808,7.475758,7.639425,7.313804,0.0,0.0,0.0,0.0,0.000000,0.000000
3,0.0,0.0,0.0,0.0,0.0,0.0,18.661525,18.112144,18.795166,18.112457,...,6.772808,7.475758,7.639425,7.313804,0.0,0.0,0.0,0.0,0.000000,0.000000
4,0.0,0.0,0.0,0.0,0.0,0.0,18.596177,18.064552,18.723999,18.038879,...,6.772808,7.475758,7.639425,7.313804,0.0,0.0,0.0,0.0,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17467,0.0,0.0,0.0,0.0,0.0,0.0,16.088258,15.146526,16.435059,14.799408,...,6.772808,7.475758,7.639425,7.313804,0.0,0.0,0.0,0.0,0.000000,0.000000
17468,0.0,0.0,0.0,0.0,0.0,0.0,16.941294,14.764474,15.987488,14.632263,...,6.772808,7.475758,7.639425,7.313804,0.0,0.0,0.0,0.0,0.000000,0.000000
17469,0.0,0.0,0.0,0.0,0.0,0.0,16.408036,14.624276,15.823792,14.117371,...,6.772808,7.475758,7.639425,7.313804,0.0,0.0,0.0,0.0,0.000000,0.000000
17470,0.0,0.0,0.0,0.0,0.0,0.0,16.804959,14.390294,15.572144,13.371262,...,6.772808,7.475758,7.639425,7.313804,0.0,0.0,0.0,0.0,0.000000,0.000000


In [10]:
class solo_models:
    def __new__(self, train_c_x, train_c_y):
        print('INITIATING PARAMETER')
        self.train_c_x, self.train_c_y= train_c_x, train_c_y
        #print(self.train_c_y.index)
        self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(train_c_x, train_c_y, test_size=0.1, random_state=200)
        #print("train_c_x:",self.train_c_x.index,"train_c_y:",self.train_c_y.index,"X_tain:",self.X_train.index)
        self.R_Forest_parm = {'n_estimators' : 25, 
                              'min_samples_split' : 2, 
                              'max_depth' : 10, 
                              'min_samples_leaf' : 2, 
                              'random_state' : 42}
        
        self.Extra_parm = {'n_estimators' : 50, 
                           'min_samples_split' : 2, 
                           'max_depth' : 8, 
                           'min_samples_leaf' : 2, 
                           'random_state' : 42}
        
        self.LGBM_R_parm = {'colsample_bytree': 0.6657998165699125,
                           'drop_rate': 0.8303473371870002,
                           'learning_rate': 0.2762739125344964,
                           'max_bin': 9983,
                           'max_depth': 8623,
                           'max_drop': 5480,
                           'min_child_samples': 101,
                           'min_data_in_leaf': 426,
                           'n_estimators': 1628,
                           'num_leaves': 3640,
                           'objective': 'regression_l1',
                           'reg_alpha': 0.39940189926691194,
                           'reg_lambda': 0.9567353299218986,
                           'skip_drop': 0.6274640597528743,
                           'verbosity': -1}
        
        self.XGB_R_parm = {'colsample_bytree' : 0.871893537724492,
                           'gamma' : 1,
                           'learning_rate' : 0.923192518624813,
                           'max_depth' : 15,
                           'min_child_weight' : 1,
                           'n_estimators' : 17748,
                           'reg_alpha' : 45,
                           'reg_lambda' : 0.8598696247943665,
                           'subsample' : 0.9109367356405415,
                           'random_state' : 42}

        self.catboost_params = {'iterations' : 3000,
                                'learning_rate': 0.009, 
                                'depth': 5, 
                                'l2_leaf_reg': 5.5,
                                'min_child_samples' : 102,
                                'od_wait' : 50,
                                'random_state' : 42,
                                'eval_metric': 'RMSE', 
                                'od_type' : 'Iter',
                                'bootstrap_type': 'Bayesian', 
                                'grow_policy' : 'Depthwise',
                                'logging_level' : 'Silent'}
        
        self.DecisionTree = {'max_depth': 6}
        self.settings = {"time_budget": 2000,
                         "metric": "mae",
                         "estimator_list": ["lgbm"],
                         "task": "regression",
                         "log_file_name": "experiment.log",
                         "seed": 41,
                         "eval_method": "cv"
                    }
        self.X_train.to_csv('data/X_train.csv', index = False)
        self.y_train.to_csv('data/y_train.csv', index = False)
        self.X_test.to_csv('data/X_test.csv', index = False)
        self.y_test.to_csv('data/y_test.csv', index = False)
        
        self.models(self)
        self.manual_training(self)
        return {'manual_ensemble' : self.model_collecter, 'stack_ensemble' : self.stack_training(self)}
        
    def models(self):
        print('LOADING MODEL')
        self.model_collecter = {}
        
        self.LGBM_R = LGBMRegressor(**self.LGBM_R_parm)
        self.XGB_R = XGBRegressor(**self.XGB_R_parm)
        self.catboost = CatBoostRegressor(**self.catboost_params)

        #self.model_collecter['random_forest'] = BaggingRegressor(estimator = RandomForestRegressor(**self.R_Forest_parm), 
        #                                                         n_estimators = 10, random_state = 0)
        #self.model_collecter['extra_trees'] = BaggingRegressor(estimator = ExtraTreesRegressor(**self.Extra_parm), 
        #                                                         n_estimators = 10, random_state = 0)
        #self.model_collecter['GradientBoostingRegressor'] = BaggingRegressor(estimator = GradientBoostingRegressor(learning_rate=0.1, 
        #                                                                                                           min_samples_split=500,
        #                                                                                                           min_samples_leaf=50,
        #                                                                                                           max_depth=8,
        #                                                                                                           max_features='sqrt',
        #                                                                                                           subsample=0.8,
        #                                                                                                           random_state=10), 
        #                                                                    n_estimators = 10, random_state = 0)
        
        #self.model_collecter['DecisionTreeRegressor'] = BaggingRegressor(estimator = DecisionTreeRegressor(**self.DecisionTree), 
        #                                                         n_estimators = 10, random_state = 0)
        #self.model_collecter['TweedieRegressor'] = BaggingRegressor(estimator = TweedieRegressor(), 
        #                                                         n_estimators = 10, random_state = 0)
        #self.model_collecter['LinearRegression'] = BaggingRegressor(estimator = LinearRegression(), 
        #                                                         n_estimators = 10, random_state = 0)
        
        #self.model_collecter['ElasticNet'] = BaggingRegressor(estimator = ElasticNetCV(), n_estimators = 10, random_state = 0)
        #self.model_collecter['LassoCV'] = BaggingRegressor(estimator = LassoCV(), n_estimators = 10, random_state = 0)
        #self.model_collecter['RidgeCV'] = BaggingRegressor(estimator = RidgeCV(), n_estimators = 10, random_state = 0)
        #self.model_collecter['LassoLars'] = BaggingRegressor(estimator = LassoLars(), n_estimators = 10, random_state = 0)
        #self.model_collecter['Lasso'] = BaggingRegressor(estimator = Lasso(), n_estimators = 10, random_state = 0)
        #self.model_collecter['BayesianRidge'] = BaggingRegressor(estimator = BayesianRidge(), n_estimators = 10, random_state = 0)
        #self.model_collecter['SVR'] = BaggingRegressor(estimator = SVR(), n_estimators = 10, random_state = 0)
        #self.model_collecter['PLSRegression'] = BaggingRegressor(estimator = PLSRegression(), n_estimators = 10, random_state = 0)
        #self.model_collecter['KMeans'] = BaggingRegressor(estimator = KMeans(), n_estimators = 10, random_state = 0)
        #self.model_collecter['GaussianProcessRegressor'] = GaussianProcessRegressor()
        #self.model_collecter['MLPRegressor'] = BaggingRegressor(estimator = MLPRegressor(), n_estimators = 10, random_state = 0)
        
    def manual_training(self):
        print('TRAINING')
        
        for i in self.model_collecter.keys():
            print(f'--{i.upper()}')
            #print(self.X_train)
            self.model_collecter[i].fit(self.X_train, self.y_train)
            joblib.dump(self.model_collecter[i], f'model/{i}.pkl')

        print('--CATBOOST')
        cat_features = list(self.X_train.select_dtypes(include=['object', 'category']).columns)
        train_pool = Pool(self.X_train, self.y_train, cat_features=cat_features)
        val_pool = Pool(self.X_test, self.y_test, cat_features=cat_features)
        
        self.catboost.fit(train_pool, eval_set=(val_pool), verbose=100, early_stopping_rounds=100)
        print('--XGBOOST')
        self.XGB_R.fit(self.X_train, self.y_train, eval_set = [(self.X_test, self.y_test)],eval_metric = "rmse", verbose = False)
        print('--LGBOOST')
        self.LGBM_R.fit(self.X_train, self.y_train,eval_set = [(self.X_test, self.y_test)],eval_metric ='rmse')
        
        self.model_collecter['LGBMRegressor'] = self.LGBM_R
        self.model_collecter['XGBRegressor'] = self.XGB_R
        self.model_collecter['CatBoostRegressor'] = self.catboost

        joblib.dump(self.model_collecter['LGBMRegressor'], f'model/LGBMRegressor.pkl')
        joblib.dump(self.model_collecter['XGBRegressor'], f'model/XGBRegressor.pkl')
        joblib.dump(self.model_collecter['CatBoostRegressor'], f'model/CatBoostRegressor.pkl')
        
    def stack_training(self):    
        print('--STACK')
        #self.model_collecter['LGBMRegressor'] = self.LGBM_R
        #self.model_collecter['XGBRegressor'] = self.XGB_R
        #self.model_collecter['CatBoostRegressor'] = self.catboost
        self.stack_model = ['LGBMRegressor',
                            'LGBMRegressor',
                            'CatBoostRegressor',
                            'random_forest',
                            'extra_trees',
                            'GradientBoostingRegressor',
                            'DecisionTreeRegressor']
        
        estimators = [(i, self.model_collecter[i]) for i in self.model_collecter.keys() if i in self.stack_model]
        self.model = StackingRegressor(estimators, final_estimator = RidgeCV())
        self.model.fit(self.X_train, self.y_train)
        joblib.dump(self.model, f'model/STACK.pkl')
        return self.model

In [11]:
E_model = solo_models(train_c_x, train_c_y)

INITIATING PARAMETER
LOADING MODEL
TRAINING
--CATBOOST
--XGBOOST
--LGBOOST
--STACK
